# Tập 4.4 · Học tăng cường: robot học từ thất bại (Q-learning) 🎮

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap44/tap44_qlearn.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 4 · ⏱ ~35 phút · Giáo trình: Principles of Robot Autonomy, [Chương 18](https://porabook.com/book/reinforcement-learning/)

Robot không biết sàn trơn bao nhiêu, tự đi thử và học. Trong notebook này bạn:

1. Xem J* khi biết mô hình (tập 4.3) để làm mốc.
2. **Tự viết** ε-greedy, cập nhật Q và chính sách học được (3 bài); ô chấm cho robot học bằng code của bạn.
3. Vẽ đường học và chính sách robot tự tìm ra.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap44")
sys.path[:0] = ["bai_tap", "sim", ".."]

sys.path.insert(0, "../tap42/sim")    # kho hàng của tập 4.2
import numpy as np
import qlearn as goc               # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"J* khi biết mô hình (quy hoạch động): {goc.M.toi_uu(goc.TRUOT)[0][goc.M.S]:.2f}")
Q, ls, _ = goc.hoc(n_tap=2000, alpha=0.2, eps=0.1, seed=0, ghi_moi=500)
for tap, J, roi, thuong in ls:
    print(f"sau {tap:4d} tập: J của chính sách tham lam {J:6.2f} · rơi {roi * 100:4.1f}% trong 500 tập gần nhất")

## 2. Bài tập

$Q(x,u) \leftarrow Q(x,u) + \alpha\big(r + \gamma\max_{u'}Q(x',u') - Q(x,u)\big)$; nếu tập đã kết thúc thì mục tiêu chỉ là $r$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `chon_hanh_dong(Qi, eps, rng)` | ε-greedy (gọi `rng` đúng thứ tự) |
| **2** | `cap_nhat_q(Q, i, a, r, i2, xong, alpha, gamma)` | cập nhật Q tại (i, a) |
| **3** | `chinh_sach_tham_lam(Q, cac_o, IDX, U)` | dict {ô: lệnh có Q lớn nhất} |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np


def chon_hanh_dong(Qi, eps, rng):
    """Bài 1 (ε-greedy): Qi là hàng Q của ô hiện tại (4 số). Gọi rng ĐÚNG thứ tự này:

        nếu rng.random() < eps:  a = int(rng.integers(4))                                  (khám phá: chọn bừa)
        ngược lại:               a = int(rng.choice(np.flatnonzero(Qi == Qi.max())))       (khai thác: chọn lệnh tốt nhất, hoà thì bốc thăm)
    Trả về a."""
    raise NotImplementedError("TODO Bài 1: ε-greedy")


def cap_nhat_q(Q, i, a, r, i2, xong, alpha, gamma):
    """Bài 2 (Eq. 18.18): cập nhật Q tại (i, a) sau khi làm lệnh a ở ô i, nhận thưởng r, tới ô i2.

        mục tiêu = r                         nếu xong (tập kết thúc: tới G hoặc rơi)
                 = r + gamma·max_a' Q[i2, a']  nếu chưa xong
        Q[i, a] ← Q[i, a] + alpha·(mục tiêu − Q[i, a])
    Sửa thẳng vào Q (không cần trả về)."""
    raise NotImplementedError("TODO Bài 2: cập nhật Q")


def chinh_sach_tham_lam(Q, cac_o, IDX, U):
    """Bài 3: chính sách học được π(x) = lệnh có Q lớn nhất (np.argmax, hoà thì lấy lệnh đứng trước).

    cac_o: list các ô; IDX: dict ô → chỉ số hàng trong Q; U: list tên lệnh. Trả về dict {ô: tên lệnh}."""
    raise NotImplementedError("TODO Bài 3: chính sách tham lam")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong cả 3 bài, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Khám phá nhiều hay ít thì tốt? Học với các ε khác nhau (3 seed mỗi ε).

In [ ]:
for eps in [0.0, 0.1, 0.3]:
    Js = [goc.hoc(n_tap=2000, eps=eps, seed=sd, alpha=0.2, ghi_moi=2000)[1][-1][1] for sd in range(3)]
    print(f"ε = {eps:.1f}: J của chính sách học được {[round(j, 2) for j in Js]}")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap44).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)